In [1]:
# %% [markdown]
# # # ETS-ANN Hybrid Model; forecast horizon = 10
# # # Python version 3.11+

# %% [markdown]
# ## 1. Import Libraries

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time

# Data and Preprocessing
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split # For tuning split

# ETS-ANN specific
# Ensure pycaret is installed: pip install pycaret[full]
# Ensure tensorflow is installed: pip install tensorflow
try:
    from pycaret.time_series import TSForecastingExperiment, setup, create_model, compare_models, predict_model, get_config
except ImportError:
    print("PyCaret time series module not found. Please install it: pip install pycaret[full]")
    exit()
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Visualization
import plotly.graph_objects as go
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# %% [markdown]
# ## 2. Configuration

# %%
# --- Defined Parameters ---
ticker = "LTC-USD"
start_date = "2017-11-09"
end_date = "2025-01-01"

# Define Train/Test Split Ratio for the *initial* training phase
train_split_ratio = 0.80
# Define Validation Split Ratio *within* the initial training residuals for tuning
validation_split_ratio_for_tuning = 0.20 # 20% of initial residuals data for validation

# --- Walk-Forward Forecast Horizon ---
h = 10 # >>> Changed to t+10 <<<
print(f"Setting Walk-Forward Horizon to: h = {h}")

# ANN Parameters
lags_ann = [1, 7, 30] # Lags used for residual prediction
max_lags = max(lags_ann)

# ANN Hyperparameter Tuning Grid
HP_ANN_NEURONS_OPTIONS = [25, 50, 100]
HP_ANN_EPOCHS_OPTIONS = [30, 50]
HP_ANN_BATCH_SIZE_OPTIONS = [32, 64]
HP_ANN_LR_TUNE = 0.001 # Learning rate for tuning

# Final ANN Training Configuration (defaults used if tuning fails)
FINAL_ANN_NEURONS_DEFAULT = 50 # Example default
FINAL_ANN_EPOCHS_DEFAULT = 50
FINAL_ANN_BATCH_SIZE_DEFAULT = 32
FINAL_ANN_LR = 0.001 # Learning rate for final training

# Retraining configuration
RETRAIN_FREQUENCY = 0 # Set > 0 to enable periodic retraining
# RETRAIN_EPOCHS_ANN = 5 # Epochs used during retraining if enabled

# %% [markdown]
# ## 3. Data Loading and Preparation

# %%
print(f"--- Loading Data for {ticker} ---")
try:
    df_full = yf.download(tickers=[ticker], start=start_date, end=end_date, progress=False)
    if df_full.empty: raise ValueError(f"No data downloaded for {ticker}.")
    if 'Close' not in df_full.columns: raise ValueError(f"'Close' column not found.")
    df_full = df_full[['Close']].copy()
    # Ensure daily frequency and forward fill missing values
    df_full = df_full.asfreq('D')
    df_full.ffill(inplace=True); df_full.dropna(inplace=True)
    if df_full.empty: raise ValueError(f"Data became empty after processing.")
    print(f"Loaded {len(df_full)} data points for {ticker} from {df_full.index.min().strftime('%Y-%m-%d')} to {df_full.index.max().strftime('%Y-%m-%d')}.")
except Exception as e:
    raise ValueError(f"Failed to load data for {ticker}: {e}")

# %% [markdown]
# ## 4. Data Splitting (Adjusted for h-step Evaluation)

# %%
# Split Data into initial training and test sets
n_total = len(df_full)
n_train = int(train_split_ratio * n_total)
# Adjust n_test: number of times we initiate an h-step forecast
# We need at least h points in the test set to evaluate the h-th step forecast
n_test = n_total - n_train - h + 1

if n_test <= 0:
     raise ValueError(f"Not enough data for walk-forward with h={h}. Need at least {n_train + h} total points.")

train_data_df = df_full[:n_train]
test_data_full = df_full[n_train:] # Holds all data from the start of the test period

print(f"\nInitial Training Data: {n_train} points ({train_data_df.index.min().strftime('%Y-%m-%d')} to {train_data_df.index.max().strftime('%Y-%m-%d')})")
print(f"Test Data Available (for updates & targets): {len(test_data_full)} points")
print(f"Number of walk-forward steps (predictions to generate & evaluate): {n_test}")
# Determine the actual evaluation period dates
if h > 0 and len(test_data_full) >= h:
    eval_start_date = test_data_full.index[h-1].strftime('%Y-%m-%d')
    eval_end_date = test_data_full.index[-1].strftime('%Y-%m-%d')
    print(f"Evaluation Period Start (target t+{h}): {eval_start_date}")
    print(f"Evaluation Period End (target t+{h}): {eval_end_date}")
else:
    print("Evaluation Period cannot be determined due to insufficient test data for horizon.")

# %% [markdown]
# ## 5. Initial ETS Model Selection and Fit

# %%
print("\n--- Initial ETS Model Training ---")
start_time_initial_ets = time.time()
exp_ets_initial = TSForecastingExperiment()
# Setup on initial training data only
# fh for setup doesn't strictly need to match h, but using a reasonable value like 30 is fine
setup(data=train_data_df, fh=min(len(test_data_full), 30),
      session_id=123, verbose=False, numeric_imputation_target="ffill")
print("Comparing ETS models...")
best_ets_model_obj = compare_models(include=['ets', 'exp_smooth'], sort='RMSE', n_select=1, verbose=False)
print(f"Initial ETS Model Selected: {best_ets_model_obj}")
# Fit the selected model on the training data
initial_ets_model_fitted = create_model(best_ets_model_obj, verbose=False)
end_time_initial_ets = time.time()
print(f"Initial ETS training finished in {end_time_initial_ets - start_time_initial_ets:.2f} seconds.")

# %% [markdown]
# ## 6. Initial Residual Calculation and Scaling

# %%
print("\n--- Calculating Initial Residuals ---")
# Get the training data used by pycaret
y_train_pycaret = get_config('y_train')
# Try to get fitted values directly, otherwise predict on train set
try:
    # Attempt to access fitted values from the underlying forecaster
    ets_fitted_values_train = initial_ets_model_fitted._fitted_forecaster.fittedvalues
    # Align index with original training data
    ets_fitted_values_train = ets_fitted_values_train.reindex(y_train_pycaret.index).dropna()
    if ets_fitted_values_train.empty: raise AttributeError # Force fallback if reindex fails
except AttributeError:
    print("Warning: Could not access fittedvalues directly. Using predict_model on training data as fallback.")
    # Fallback: Use predict_model on the training data
    ets_fitted_values_train = predict_model(initial_ets_model_fitted, data=y_train_pycaret)['y_pred']
    ets_fitted_values_train = ets_fitted_values_train.reindex(y_train_pycaret.index).dropna()

# Ensure indices align before subtraction
y_train_pycaret_aligned = y_train_pycaret.reindex(ets_fitted_values_train.index)

residuals_initial_train = y_train_pycaret_aligned - ets_fitted_values_train
residuals_initial_train.dropna(inplace=True) # Drop NaNs potentially introduced by alignment/fitting start
print(f"Calculated {len(residuals_initial_train)} initial residuals.")

if len(residuals_initial_train) == 0:
    raise ValueError("No initial residuals could be calculated. Check ETS model fitting and data alignment.")

print("\n--- Scaling Initial Residuals ---")
scaler_residuals = MinMaxScaler(feature_range=(-1, 1)) # Scale residuals to [-1, 1]
normalized_residuals_initial_train = scaler_residuals.fit_transform(residuals_initial_train.values.reshape(-1, 1))
# Create a pandas Series for easier lagging later
normalized_residuals_initial_train_series = pd.Series(normalized_residuals_initial_train.flatten(), index=residuals_initial_train.index)
print("Residual scaler fitted on initial training residuals.")

# %% [markdown]
# ## 7. Prepare Lagged Residual Data for ANN

# %%
print("\n--- Preparing Lagged Residual Features ---")
def create_lagged_features_ann(series, lags):
    """Creates DataFrame with target and lagged features for ANN."""
    lagged_data = pd.DataFrame(index=series.index)
    lagged_data['target'] = series # Current normalized residual is the target
    for lag in lags:
        lagged_data[f'lag_{lag}'] = series.shift(lag)
    lagged_data.dropna(inplace=True) # Drop rows with NaNs due to shifting
    return lagged_data

# Create lagged features from the normalized initial training residuals
lagged_norm_resid_initial = create_lagged_features_ann(normalized_residuals_initial_train_series, lags_ann)

if lagged_norm_resid_initial.empty:
     raise ValueError(f"Lagged residual dataset is empty. Initial residuals might be too short for max lag ({max_lags}).")

X_ann_initial_features = lagged_norm_resid_initial.drop('target', axis=1)
y_ann_initial_target = lagged_norm_resid_initial['target']
print(f"Created lagged residual dataset with shape: Features={X_ann_initial_features.shape}, Target={y_ann_initial_target.shape}")

# %% [markdown]
# ## 8. ANN Hyperparameter Tuning (on Initial Residuals)

# %%
print("\n--- Starting ANN Hyperparameter Tuning ---")
start_time_tuning = time.time()
# Split the initial *lagged residual data* for tuning
# Important: shuffle=False preserves time order if needed, though less critical for feedforward ANN on residuals
X_ann_train_tune, X_ann_val_tune, y_ann_train_tune, y_ann_val_tune = train_test_split(
    X_ann_initial_features, y_ann_initial_target,
    test_size=validation_split_ratio_for_tuning,
    shuffle=False # Keep temporal order for validation split
)
print(f"ANN Tuning Train shape: {X_ann_train_tune.shape}, Validation shape: {X_ann_val_tune.shape}")

best_val_mse_tune = float('inf')
best_params_ann = None

# Manual Grid Search loop
for neurons in HP_ANN_NEURONS_OPTIONS:
    for epochs in HP_ANN_EPOCHS_OPTIONS:
        for batch_size in HP_ANN_BATCH_SIZE_OPTIONS:
            # print(f"Tuning Trial: Neurons={neurons}, Epochs={epochs}, Batch Size={batch_size}") # Verbose tuning log

            # Build ANN model for tuning trial
            ann_model_tune = Sequential([
                Dense(neurons, activation='relu', input_shape=(X_ann_train_tune.shape[1],)), # Input shape based on number of lags
                Dense(max(10, neurons//2), activation='relu'), # Example hidden layer
                Dense(1) # Single output neuron for residual prediction
            ])
            ann_model_tune.compile(optimizer=Adam(learning_rate=HP_ANN_LR_TUNE), loss='mse') # Mean Squared Error for regression

            # Train model on tuning training set, validate on tuning validation set
            history = ann_model_tune.fit(X_ann_train_tune.values, y_ann_train_tune.values,
                                         epochs=epochs,
                                         batch_size=batch_size,
                                         validation_data=(X_ann_val_tune.values, y_ann_val_tune.values),
                                         verbose=0) # Suppress epoch output during tuning

            # Evaluate performance on tuning validation set
            if 'val_loss' in history.history and len(history.history['val_loss']) > 0:
                 # Get the final validation loss for this trial
                 val_mse = history.history['val_loss'][-1]
                 # print(f"  Validation MSE: {val_mse:.6f}") # Verbose tuning log
                 if val_mse < best_val_mse_tune:
                     best_val_mse_tune = val_mse
                     # Store the best parameters found so far
                     best_params_ann = {'neurons': neurons, 'epochs': epochs, 'batch_size': batch_size}
            else:
                 print("  Warning: No validation loss recorded for this trial.")

end_time_tuning = time.time()
print(f"\n--- ANN Tuning Complete in {end_time_tuning - start_time_tuning:.2f} seconds ---")
if best_params_ann is None:
     print("Warning: ANN Tuning failed to find best parameters. Using defaults.")
     # Use defaults defined in Configuration if tuning fails
     best_params_ann = {'neurons': FINAL_ANN_NEURONS_DEFAULT, 'epochs': FINAL_ANN_EPOCHS_DEFAULT, 'batch_size': FINAL_ANN_BATCH_SIZE_DEFAULT}
else:
     print(f"Best Hyperparameters found: {best_params_ann}")
     print(f"Best Validation MSE during tuning: {best_val_mse_tune:.6f}")

# %% [markdown]
# ## 9. Train Final Initial ANN Model

# %%
print("\n--- Training Final Initial ANN Model on Residuals ---")
start_time_initial_ann = time.time()

# Build the final initial ANN model using the best found hyperparameters
final_ann_model = Sequential([
    Dense(best_params_ann['neurons'], activation='relu', input_shape=(X_ann_initial_features.shape[1],)),
    Dense(max(10, best_params_ann['neurons']//2), activation='relu'),
    Dense(1)
])
final_ann_model.compile(optimizer=Adam(learning_rate=FINAL_ANN_LR), loss='mse') # Use final LR

print(f"Training final initial ANN with {best_params_ann['neurons']} neurons for {best_params_ann['epochs']} epochs...")
# Train on the *entire* initial lagged residual dataset
final_ann_model.fit(X_ann_initial_features.values, y_ann_initial_target.values,
                    epochs=best_params_ann['epochs'],
                    batch_size=best_params_ann['batch_size'],
                    verbose=0) # Suppress epoch output for final training

end_time_initial_ann = time.time()
print(f"Final Initial ANN training complete in {end_time_initial_ann - start_time_initial_ann:.2f} seconds.")
final_ann_model.summary()

# %% [markdown]
# ## 10. Prepare for Walk-Forward Loop

# %%
print("\n--- Preparing for Walk-Forward ---")
# Get the underlying fitted forecaster object from PyCaret
try:
    # Accessing internal attribute - might change in future pycaret versions
    fitted_ets_forecaster = initial_ets_model_fitted._fitted_forecaster
    print(f"Using underlying forecaster: {type(fitted_ets_forecaster)}")
except AttributeError:
    raise AttributeError("Could not access the underlying '_fitted_forecaster' object from the PyCaret model. Check PyCaret version or model structure.")

# Initialize history (starts with initial training residuals)
history_norm_residuals = normalized_residuals_initial_train_series.tolist()
# Pad history if shorter than max_lag needed for ANN input
if len(history_norm_residuals) < max_lags:
     print(f"Warning: Padding initial residual history from {len(history_norm_residuals)} to {max_lags} with zeros.")
     # Pad with zeros at the beginning
     history_norm_residuals = [0.0] * (max_lags - len(history_norm_residuals)) + history_norm_residuals
print(f"Initial normalized residual history length: {len(history_norm_residuals)}")

# List to store final h-step ahead hybrid predictions
ets_ann_walk_forward_predictions_h_step = []

# %% [markdown]
# ## 11. Walk-Forward Validation (Rolling Forecast) Loop - t+h Steps Ahead

# %%
print(f"\n--- Starting ETS-ANN Walk-Forward Validation for {n_test} steps (Predicting {h} steps ahead) ---")
start_time_walk_forward = time.time()

# Indices of the test data points we need to iterate through for updates
test_indices_for_loop = test_data_full.index[:n_test]

# The main history of *actual* normalized residuals observed so far
# Starts with residuals from the initial training period
current_history_norm_residuals = list(history_norm_residuals) # Use a copy to avoid modifying initial list if run multiple times

for i, current_loop_date in enumerate(test_indices_for_loop):

    # Index in the FULL dataset for the *last known actual* data point
    # before this iteration's multi-step prediction starts
    current_actual_index = n_train + i - 1 # Index of data point at time t-1 (or end of train)
    if i == 0:
        current_actual_index = n_train - 1 # For the first iteration, last known is end of training

    # --- ETS Prediction (Component 1: Predict t+1 to t+h) ---
    ets_forecast_h_steps = np.full(h, np.nan) # Initialize forecast array
    try:
        # Predict h steps starting *after* current_actual_index
        predict_start_index = current_actual_index + 1
        predict_end_index = current_actual_index + h
        # Use the predict method of the underlying forecaster object
        ets_forecast_h_steps = fitted_ets_forecaster.predict(fh=np.arange(1, h + 1)) # Predict relative fh
        # Ensure the output is a numpy array of length h
        if isinstance(ets_forecast_h_steps, pd.Series):
            ets_forecast_h_steps = ets_forecast_h_steps.values
        if len(ets_forecast_h_steps) != h:
             # If prediction failed or returned wrong length, use naive persistence as fallback
            print(f"Warning: ETS predict length mismatch step {i+1}. Got {len(ets_forecast_h_steps)}, expected {h}. Using naive fallback.")
            last_known_price = df_full['Close'].iloc[current_actual_index]
            ets_forecast_h_steps = np.full(h, last_known_price)

    except Exception as e:
        print(f"Warning: ETS predict failed step {i+1}. Error: {e}. Using naive fallback.")
        last_known_price = df_full['Close'].iloc[current_actual_index]
        ets_forecast_h_steps = np.full(h, last_known_price)


    # --- ANN Residual Prediction (Component 2: Predict t+1 to t+h iteratively) ---
    ann_pred_h_steps_denorm = np.zeros(h) # Store denormalized residual forecasts
    # Use a temporary history for iterative ANN prediction to not pollute main history
    temp_history_for_ann_pred = list(current_history_norm_residuals)

    for step_h in range(h): # Inner loop for iterative prediction
        if len(temp_history_for_ann_pred) >= max_lags:
            # Prepare input vector using the most recent lags from temp history
            current_input_features = [temp_history_for_ann_pred[-lag] for lag in lags_ann]
            input_vector = np.array(current_input_features).reshape(1, -1)

            # Predict the *next* normalized residual
            ann_pred_next_norm = final_ann_model.predict(input_vector, verbose=0)[0, 0]

            # Denormalize this prediction for storage
            ann_pred_next_denorm = scaler_residuals.inverse_transform([[ann_pred_next_norm]])[0, 0]
            ann_pred_h_steps_denorm[step_h] = ann_pred_next_denorm

            # Append the *predicted normalized* residual to the temporary history
            # This is the crucial step for iterative forecasting
            temp_history_for_ann_pred.append(ann_pred_next_norm)
        else:
            # Should not happen if initial history padding was correct, but good safeguard
            print(f"Warning: Not enough history for ANN prediction at inner step {step_h+1} (main step {i+1}). Setting remaining residual forecasts to 0.")
            ann_pred_h_steps_denorm[step_h:] = 0 # Assume zero residual if history fails
            break # Exit inner loop

    # --- Combine Forecasts & Store Target h-step Prediction ---
    # The target forecast is the sum of the h-th ETS forecast and the h-th ANN residual forecast
    final_pred_t_plus_h = ets_forecast_h_steps[h-1] + ann_pred_h_steps_denorm[h-1]
    ets_ann_walk_forward_predictions_h_step.append(final_pred_t_plus_h)

    # --- Update *Main* History with ACTUAL value/residual for step `t` ---
    # This uses the actual price that became available at the end of iteration `t`
    update_actual_index = n_train + i # Index of the actual value Y(t)
    actual_price_t = df_full['Close'].iloc[update_actual_index]

    # Calculate the ACTUAL residual for this step 't'
    try:
        # Get the ETS forecast that *would have been made for step t* using data up to t-1
        ets_forecast_t = fitted_ets_forecaster.predict(fh=[1])[0] # Predict 1 step from previous state
    except Exception as e_ets_t:
        print(f"Warning calculating actual residual: ETS predict failed ({e_ets_t}). Using naive fallback for residual calc.")
        # Fallback: use previous ETS forecast component or last known price
        if i == 0:
             # Use last fitted value from initial training if available
             try: ets_forecast_t = ets_fitted_values_train.iloc[-1]
             except NameError: ets_forecast_t = df_full['Close'].iloc[n_train - 1] # Ultimate fallback
        else:
             # Use the t+1 ETS forecast made in the *current* iteration as proxy
             ets_forecast_t = ets_forecast_h_steps[0]

    actual_residual_t = actual_price_t - ets_forecast_t
    try:
        actual_residual_float = float(actual_residual_t)
        # Normalize the actual residual using the scaler fitted on initial residuals
        actual_residual_t_norm = scaler_residuals.transform([[actual_residual_float]])[0, 0]
    except Exception as e_transform:
        print(f"ERROR transforming actual residual at step {i+1}: {e_transform}. Using 0.0 for history update.")
        actual_residual_t_norm = 0.0 # Avoid appending NaN if possible

    # Append the *actual* normalized residual to the main history for the next iteration
    current_history_norm_residuals.append(actual_residual_t_norm)

    # Optional: Log progress (adjust frequency for potentially long runs)
    if (i + 1) % 50 == 0 or (i+1) == n_test: # Log every 50 steps or at the end
        print(f"ETS-ANN Walk-Forward (h={h}) Step {i+1}/{n_test} complete.")

    # --- Optional: Periodic Retraining ---
    # if RETRAIN_FREQUENCY > 0 and (i + 1) % RETRAIN_FREQUENCY == 0 and (i + 1) < n_test:
        # Implement full refitting of ETS and ANN on data up to current_actual_index + 1
        # This would be computationally very expensive
        # print(f"\n--- Retraining ETS-ANN at step {i+1}/{n_test} ---")
        # ... (retraining logic would go here) ...

end_time_walk_forward = time.time()
total_walk_forward_time = end_time_walk_forward - start_time_walk_forward
print(f"\nETS-ANN Walk-Forward (h={h}) finished in {total_walk_forward_time:.2f} seconds.")

ets_ann_walk_forward_predictions_h_step = np.array(ets_ann_walk_forward_predictions_h_step)
print(f"Length of final predictions generated: {len(ets_ann_walk_forward_predictions_h_step)}") # Should be n_test
print(f"Number of walk-forward steps performed: {n_test}") # Check consistency

# %% [markdown]
# ## 12. Evaluate Walk-Forward Performance (t+h)

# %%
# Define evaluation metrics function (reusable)
def evaluate_forecast(y_true, y_pred, model_name, horizon):
    """Calculates and prints standard evaluation metrics."""
    # Ensure inputs are flat numpy arrays
    y_true_flat = np.array(y_true).flatten()
    y_pred_flat = np.array(y_pred).flatten()

    # Basic check for NaNs/Infs
    valid_indices = ~np.isnan(y_true_flat) & ~np.isnan(y_pred_flat) & ~np.isinf(y_true_flat) & ~np.isinf(y_pred_flat)
    y_true_clean = y_true_flat[valid_indices]
    y_pred_clean = y_pred_flat[valid_indices]

    if len(y_true_clean) == 0:
        print(f"\n--- {model_name} Walk-Forward (t+{horizon}) Evaluation Results ---")
        print("Evaluation skipped: No valid points after cleaning.")
        return {'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan, 'R2': np.nan}

    mae = mean_absolute_error(y_true_clean, y_pred_clean)

    # Handle potential zeros in actuals for MAPE
    mask = y_true_clean != 0
    if np.any(mask):
        mape = np.mean(np.abs((y_true_clean[mask] - y_pred_clean[mask]) / y_true_clean[mask]))
    else:
        mape = np.nan

    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    try:
        if np.var(y_true_clean) < 1e-9:
             r2 = np.nan
             print("Warning: Constant actual values detected. R² score is NaN.")
        else:
             r2 = r2_score(y_true_clean, y_pred_clean)
    except ValueError:
        r2 = np.nan

    print(f"\n--- {model_name} Walk-Forward (t+{horizon}) Evaluation Results ---")
    print(f"RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape:.4%}, R²: {r2:.4f}")
    print(f"Number of evaluation points: {len(y_true_clean)}")
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

# Evaluate against the actual unscaled test data, shifted by h-1 steps
# The prediction made at step i corresponds to the actual value at step i + h - 1 in the test set
start_actual_idx_eval = n_train + h - 1
end_actual_idx_eval = n_train + n_test + h -1 # Index of the last target value
end_actual_idx_eval = min(end_actual_idx_eval, n_total) # Ensure not out of bounds

y_test_actual_h_step = df_full['Close'].values[start_actual_idx_eval : end_actual_idx_eval]

# Check lengths before evaluation
if len(y_test_actual_h_step) != len(ets_ann_walk_forward_predictions_h_step):
     # This can happen if the loop stopped early or index calc is off
     print(f"Warning: Length mismatch during evaluation. Actuals={len(y_test_actual_h_step)}, Preds={len(ets_ann_walk_forward_predictions_h_step)}. Adjusting...")
     min_eval_len = min(len(y_test_actual_h_step), len(ets_ann_walk_forward_predictions_h_step))
     y_test_actual_h_step = y_test_actual_h_step[:min_eval_len]
     ets_ann_walk_forward_predictions_h_step_eval = ets_ann_walk_forward_predictions_h_step[:min_eval_len]
else:
    ets_ann_walk_forward_predictions_h_step_eval = ets_ann_walk_forward_predictions_h_step


ets_ann_wf_h_results = evaluate_forecast(y_test_actual_h_step, ets_ann_walk_forward_predictions_h_step_eval, f"ETS-ANN ({ticker})", horizon=h)

# %% [markdown]
# ## 13. Visualize Walk-Forward Results (t+h)

# %%
print("\n--- Plotting Walk-Forward Forecasts ---")

# Get the corresponding dates for the actual values being evaluated
prediction_dates = df_full.index[start_actual_idx_eval : end_actual_idx_eval]

# Use the potentially adjusted lengths from evaluation
if len(prediction_dates) != len(ets_ann_walk_forward_predictions_h_step_eval):
     print(f"Warning: Date length mismatch for plotting. Using {len(ets_ann_walk_forward_predictions_h_step_eval)} points.")
     prediction_dates = prediction_dates[:len(ets_ann_walk_forward_predictions_h_step_eval)]
     plot_actuals = y_test_actual_h_step # Already sliced during eval if needed
     plot_predictions = ets_ann_walk_forward_predictions_h_step_eval
else:
     plot_actuals = y_test_actual_h_step
     plot_predictions = ets_ann_walk_forward_predictions_h_step_eval


if len(prediction_dates) > 0:
    results_df_wf = pd.DataFrame({
        'Actual': plot_actuals.flatten(),
        f'ETS-ANN (t+{h})': plot_predictions.flatten()
    }, index=prediction_dates)

    fig = go.Figure()
    # Plot actuals for the evaluation period
    fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf['Actual'], mode='lines', name='Actual Price (Eval Period)', line=dict(color='black')))
    # Plot the h-step ahead forecasts
    fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf[f'ETS-ANN (t+{h})'], mode='lines', name=f'ETS-ANN Walk-Forward (t+{h})', line=dict(color='purple', dash='dot'))) # Changed color

    fig.update_layout(
        title=f'ETS-ANN Walk-Forward (t+{h}) Forecast Comparison for {ticker} (Tuned ANN)',
        xaxis_title="Date (Target Date of Forecast)",
        yaxis_title="Price (USD)",
        legend_title="Data/Model",
        template="plotly_white"
    )
    fig.show()
else:
    print("No evaluation points to plot.")


# %% [markdown]
# ## 14. Walk-Forward Evaluation Period Summary

# %%
print(f"\n--- Walk-Forward Evaluation Summary ---")
print(f"Initial Training Data End Date: {train_data_df.index.max().strftime('%Y-%m-%d')}")
print(f"Walk-Forward Evaluation Period (Test Set Dates for Updates): {test_data_full.index.min().strftime('%Y-%m-%d')} to {test_data_full.index.max().strftime('%Y-%m-%d')}")
print(f"Number of Walk-Forward Steps Performed: {n_test}")
print(f"Forecast Horizon Evaluated at each Step: h = {h}")
# Ensure indices exist before formatting dates for eval period target dates
if h > 0 and len(test_data_full) >= h:
    print(f"Evaluation Period (Target Dates): {test_data_full.index[h-1].strftime('%Y-%m-%d')} to {test_data_full.index[-1].strftime('%Y-%m-%d')}")
else:
    print("Evaluation Period cannot be determined due to insufficient test data for horizon.")

Setting Walk-Forward Horizon to: h = 10
--- Loading Data for LTC-USD ---
YF.download() has changed argument auto_adjust default to True
Loaded 2610 data points for LTC-USD from 2017-11-09 to 2024-12-31.

Initial Training Data: 2088 points (2017-11-09 to 2023-07-28)
Test Data Available (for updates & targets): 522 points
Number of walk-forward steps (predictions to generate & evaluate): 513
Evaluation Period Start (target t+10): 2023-08-07
Evaluation Period End (target t+10): 2024-12-31

--- Initial ETS Model Training ---
Comparing ETS models...
Initial ETS Model Selected: AutoETS(seasonal='mul', sp=6, trend='add')
Initial ETS training finished in 8.92 seconds.

--- Calculating Initial Residuals ---
Calculated 2058 initial residuals.

--- Scaling Initial Residuals ---
Residual scaler fitted on initial training residuals.

--- Preparing Lagged Residual Features ---
Created lagged residual dataset with shape: Features=(2028, 3), Target=(2028,)

--- Starting ANN Hyperparameter Tuning ---
A

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_36 (Dense)                │ (None, 25)             │           100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 12)             │           312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 1)              │            13 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,277 (4.99 KB)

 Trainable params: 425 (1.66 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 852 (3.33 KB)


--- Preparing for Walk-Forward ---
Using underlying forecaster: <class 'statsmodels.tsa.exponential_smoothing.ets.ETSResultsWrapper'>
Initial normalized residual history length: 2058

--- Starting ETS-ANN Walk-Forward Validation for 513 steps (Predicting 10 steps ahead) ---
Warning calculating actual residual: ETS predict failed (ETSResults.predict() got an unexpected keyword argument 'fh'). Using naive fallback for residual calc.
Warning calculating actual residual: ETS predict failed (ETSResults.predict() got an unexpected keyword argument 'fh'). Using naive fallback for residual calc.
Warning calculating actual residual: ETS predict failed (ETSResults.predict() got an unexpected keyword argument 'fh'). Using naive fallback for residual calc.
Warning calculating actual residual: ETS predict failed (ETSResults.predict() got an unexpected keyword argument 'fh'). Using naive fallback for residual calc.
Warning calculating actual residual: ETS predict failed (ETSResults.predict() got an


--- Walk-Forward Evaluation Summary ---
Initial Training Data End Date: 2023-07-28
Walk-Forward Evaluation Period (Test Set Dates for Updates): 2023-07-29 to 2024-12-31
Number of Walk-Forward Steps Performed: 513
Forecast Horizon Evaluated at each Step: h = 10
Evaluation Period (Target Dates): 2023-08-07 to 2024-12-31
